In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig, AutoModelForCausalLM
import faiss
from datasets import load_dataset

/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. Tokenizer 및 Meta-Llama-3.1-8B-Instruct 인코더 모델 로드
tokenizer = AutoTokenizer.from_pretrained("McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp")

# padding token이 없어서 eos_token을 padding token으로 설정
tokenizer.pad_token = tokenizer.eos_token

# Quantization 설정 (4-bit)
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# Quantized 인코더 모델 로드
model = AutoModel.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

# 모델을 명시적으로 .to(device)로 옮길 필요 없음, 이미 올바른 디바이스로 할당됨
model.eval()  # 평가 모드로 전환

In [3]:
# 2. 문서 데이터셋 준비 (Hugging Face에서 로드)
dataset = load_dataset('ag_news', split='train[:250]')  # 예시로 1000개 뉴스 사용
corpus = dataset['text']

In [5]:
# 3. LLM2Vec Llama 모델을 사용한 문서 임베딩을 계산하는 함수
def embed(texts):
    # 텍스트를 토큰화하고 패딩 및 잘림 처리
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    
    # 토큰화된 텍스트를 GPU로 보내기
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    
    # 모델을 이용해 임베딩 계산
    with torch.no_grad():
        outputs = model(**inputs)
        # 인코더의 마지막 히든 상태를 평균하여 임베딩 벡터 생성
        embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    
    return embeddings

In [ ]:
# 4. 코퍼스에 대한 임베딩 계산
corpus_embeddings = embed(corpus)

In [7]:
# 5. FAISS 인덱스 생성 및 문서 추가
index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

In [ ]:
# 5. 쿼리 예시
query = "Recent advancements in AI technology"

# 쿼리 임베딩 계산
query_embedding = embed([query])

# FAISS에서 가장 가까운 문서 검색
D, I = index.search(query_embedding, k=5)  # 가장 가까운 5개 문서 검색

# 검색된 문서 출력
retrieved_docs = [corpus[i] for i in I[0]]
print("Retrieved Documents: ", retrieved_docs)

In [ ]:
# Tokenizer 및 텍스트 생성 모델 로드 (Meta-Llama-3.1)
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
model_gen = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

In [ ]:
# padding token이 없으면 eos_token을 padding token으로 설정
tokenizer_gen.pad_token = tokenizer_gen.eos_token

# GPU 사용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gen.to(device)
model_gen.eval()    # 생성 모델을 평가 모드로 전환

In [34]:
# 6. 검색 문서를 바탕으로 쿼리와 결합하여 생성
def combine_query_and_docs(query, retrieved_docs):
    prompt = f"Question: {query}\n"
    prompt += f"The following context is retrieved from relevant articles:\n"
    prompt += "\n".join([f"Document {i+1}: {doc}" for i, doc in enumerate(retrieved_docs)])
    prompt += """Based on the above context, \
        1. Does it meet the intent of the question?\
        2. Are there any inaccurate contents?\
        3. Is there any useless information included?\
        4. Are the key points clearly revealed?\
        Generate one sentence that satisfies these four criteria.\
        The conclusion begins with "The Best answer is:"."""
    return prompt

input_text = combine_query_and_docs(query, retrieved_docs)

# 입력 텍스트를 토크나이즈하고 모델로 생성 요청 (Meta-Llama-3.1 모델 사용)
inputs = tokenizer_gen(input_text, return_tensors="pt", truncation=True).to(device)

# 명시적으로 attention_mask를 추가
inputs['attention_mask'] = (inputs['input_ids'] != tokenizer_gen.pad_token_id).long().to(device)

# 답변 생성
with torch.no_grad():  # 그래디언트 계산 비활성화
    generated = model_gen.generate(
        inputs.input_ids, 
        attention_mask=inputs.attention_mask,  # attention_mask 추가
        pad_token_id=tokenizer_gen.pad_token_id,  # pad_token_id를 명시적으로 설정
        #do_sample=True,
        #min_length=10,
        max_new_tokens=256,
        #repetition_penalty=1.5,
        #no_repeat_ngram_size=3,
        #temperature=0.9,
        #op_k=50,
        #top_p=0.92,
        # early_stopping=True
    )

In [ ]:
len(inputs.input_ids[0])

In [ ]:
generated

In [ ]:
generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)
generated_text

In [ ]:
# 생성된 텍스트 디코딩
generated_text = tokenizer.decode(generated[:, inputs.input_ids.shape[1]:][0], skip_special_tokens=True)
# -->> 입력 프롬프트를 제외하는 방법 찾아야 함


# 특정 패턴("The best answer was:")을 제외하고 답변만 추출
# "The best answer is:" 이후의 텍스트만 추출
if "The Best answer is:" in generated_text:
    generated_text = generated_text.split("The Best answer is:")[-1].strip()

# 결과 출력
print("Answer : ", generated_text)

In [ ]:
# 페이지 단위로 출력을 나누어 보여주는 함수
def paginate_output(text, chars_per_page=500):
    for i in range(0, len(text), chars_per_page):
        print(text[i:i+chars_per_page])
        input("\n--- Press Enter to see more ---\n")  # 페이지 전환
    print("\n--- End of Text ---")

# 페이지 단위로 출력
paginate_output(generated_text)

In [ ]:
# 출력 텍스트에서 원하는 부분만 추출
filtered_text = generated_text.split('.')[0].rstrip()
print("Final Text: ", filtered_text)

In [ ]:
filtered_text.split()

In [ ]:
from rouge import Rouge

# Reference answer - replace this with actual reference answers you want to compare against
reference_answer = "Recent advancements in AI technology are seen in various fields including autonomous systems and media applications."

# ROUGE 점수 계산
rouge = Rouge()
rouge_scores = rouge.get_scores(filtered_text, reference_answer)[0]
print(f"ROUGE Score: {rouge_scores}")
# ROUGE 점수에서 각 메트릭을 출력
print(f"ROUGE-1: {rouge_scores['rouge-1']['f']:.4f}")
print(f"ROUGE-2: {rouge_scores['rouge-2']['f']:.4f}")
print(f"ROUGE-L: {rouge_scores['rouge-l']['f']:.4f}")

In [ ]:
import sacrebleu

# BLEU 점수 계산
bleu_score = sacrebleu.raw_corpus_bleu([filtered_text], [[reference_answer]])
print(f"BLEU Score: {bleu_score.score:.4f}")

In [ ]:
from nltk.translate.meteor_score import meteor_score

# METEOR 점수 계산
meteor_score_value = meteor_score(reference_answer, filtered_text)
print(f"METEOR Score: {meteor_score_value:.4f}")

In [ ]:
# TER 점수 계산 (sacrebleu에 포함된 기능 사용)
ter_score = sacrebleu.TER([filtered_text], [[reference_answer]])
print(f"TER Score: {ter_score}")